### Environment check

Import the required modules and displays the Python
interpreter, Python version, working directory, and pandas version.

In [31]:
import sys
import os
import pandas as pd

print(sys.executable)
print(sys.version)
print(os.getcwd())
print(pd.__version__)

c:\Users\henri\AppData\Local\Programs\Python\Python314\python.exe
3.14.10 (tags/v3.14.10:b703b2c, Oct 25 2026, 17:36:52) [MSC v.1944 64 bit (AMD64)]
d:\IMS\Data Mining Project\notebooks
3.1.1


### Data loading and initial dimensions

The CSV is loaded into the DataFrame `data`.
The loading options preserve the literal education category "None"
while treating empty fields as missing values.

The imported dataset contains 649 rows and 34 columns.
One column is an exported row index, so these are not 34 analytical features.

In [32]:
data = pd.read_csv(
    '../data/raw/student_data.csv',
    keep_default_na=False,
    na_values=['']
)

data.shape

display(data.head())

,Unnamed: 0,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,0,GP,F,18,U,7.0,A,Higher Education,Higher Education,at_home,...,4.0,3.0,4.0,1.0,1.0,3.0,4.0,0,11,11
1,1,GP,F,17,U,4.0,T,Primary Education,Primary Education,at_home,...,5.0,3.0,3.0,1.0,1.0,3.0,2.0,9,11,11
2,2,GP,F,15,U,1.0,T,Primary Education,Primary Education,at_home,...,4.0,3.0,2.0,2.0,NaN,3.0,6.0,12,13,12
3,3,GP,F,15,U,4.0,T,Higher Education,Middle School Education,health,...,3.0,2.0,2.0,1.0,1.0,5.0,NaN,14,14,14
4,4,GP,F,16,U,9.0,T,Secondary Education,Secondary Education,other,...,4.0,3.0,2.0,1.0,2.0,5.0,NaN,11,13,13


---

### Missing-value analysis

Missing values are counted for each column to assess data completeness.
The distribution of recorded values is also inspected, including missing
entries, to support later preprocessing decisions.

In [33]:
missing_counts = data.isna().sum()
missing_percentages = (missing_counts / len(data) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing count': missing_counts,
    'Missing (%)': missing_percentages
})

missing_summary

,Missing count,Missing (%)
Unnamed: 0,0,0.00
school,1,0.15
sex,10,1.54
age,0,0.00
address,17,2.62
famsize,3,0.46
Pstatus,28,4.31
Medu,27,4.16
Fedu,19,2.93
Mjob,6,0.92


### Column types and structure

Inspect each column's stored data type and number of non-missing
values before reviewing variable roles and distributions.

In [34]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 649 entries, 0 to 648
Data columns (total 34 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  649 non-null    int64  
 1   school      648 non-null    str    
 2   sex         639 non-null    str    
 3   age         649 non-null    int64  
 4   address     632 non-null    str    
 5   famsize     646 non-null    float64
 6   Pstatus     621 non-null    str    
 7   Medu        622 non-null    str    
 8   Fedu        630 non-null    str    
 9   Mjob        643 non-null    str    
 10  Fjob        631 non-null    str    
 11  reason      642 non-null    str    
 12  guardian    641 non-null    str    
 13  traveltime  638 non-null    float64
 14  studytime   635 non-null    float64
 15  failures    98 non-null     float64
 16  schoolsup   619 non-null    str    
 17  famsup      642 non-null    str    
 18  paid        618 non-null    str    
 19  activities  618 non-null    str    
 20 

### Duplicate records

Records are checked for identical values across the student variables.
The exported row index is excluded because its unique values could
hide otherwise identical records.

In [35]:
student_data = data.drop(columns=['Unnamed: 0'])

duplicate_count = student_data.duplicated().sum()
duplicate_count

np.int64(0)

### Numerical summary

Summary statistics are inspected to understand the distribution and
range of numerical columns and identify values requiring investigation.

In [36]:
student_data.describe()

,age,famsize,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
count,649.000000,646.000000,638.000000,635.000000,98.000000,625.00000,629.000000,618.000000,625.000000,622.000000,632.000000,389.000000,649.000000,649.000000,649.000000
mean,16.744222,4.978328,18.213166,242.206299,1.428571,3.92480,3.193959,3.192557,1.508800,2.284566,3.526899,5.889460,11.399076,11.570108,11.906009
std,1.218138,2.817359,19.951079,235.969842,0.717958,0.95698,1.051490,1.179296,0.930434,1.287017,1.451565,4.682234,2.745265,2.913639,3.230656
min,15.000000,0.000000,0.000000,0.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
25%,16.000000,3.000000,6.000000,99.500000,1.000000,4.00000,3.000000,2.000000,1.000000,1.000000,2.000000,2.000000,10.000000,10.000000,10.000000
50%,17.000000,5.000000,13.000000,186.000000,1.000000,4.00000,3.000000,3.000000,1.000000,2.000000,4.000000,4.000000,11.000000,11.000000,12.000000
75%,18.000000,7.000000,25.000000,282.500000,2.000000,5.00000,4.000000,4.000000,2.000000,3.000000,5.000000,8.000000,13.000000,13.000000,14.000000
max,22.000000,9.000000,144.000000,1465.000000,3.000000,5.00000,5.000000,5.000000,5.000000,5.000000,5.000000,32.000000,19.000000,19.000000,19.000000


### Categorical summary

Categorical columns are inspected to identify the number of distinct
categories, the most frequent category, and its frequency.

In [37]:
student_data.describe(include=['str']).T

,count,unique,top,freq
school,648,2,GP,422
sex,639,2,F,377
address,632,2,U,443
Pstatus,621,2,T,544
Medu,622,5,Middle School Education,179
Fedu,630,5,Middle School Education,203
Mjob,643,5,other,254
Fjob,631,5,other,358
reason,642,4,course,279
guardian,641,3,mother,450


---

### Family size and parental status

Family size and parental status are examined together to investigate
records with a family size of zero and assess possible inconsistencies.

In [38]:
zero_family = student_data.loc[
    student_data['famsize'] == 0,
    ['famsize', 'Pstatus']
]

zero_family['Pstatus'].value_counts(dropna=False)

Pstatus
T      41
A      13
NaN     3
Name: count, dtype: int64

In [64]:
display(data.loc[data["famsize"]. == 0])

SyntaxError: invalid syntax (199399419.py, line 1)

Chatgpt: My leading hypothesis is that zero represents an invalid or undocumented value in famsize. If family size counts household members including the student, its minimum should be one.

Possible explanations include
- Zero was used as a placeholder for unknown family size.
- The column was incorrectly transformed or generated during dataset preparation.
- The intended definition differs from the assignment’s description, such as counting other family members.
The 57 zeros—8.8% of students—suggest a recurring coding issue worth investigating. Having 41 of these students marked “Together” strengthens the question, although parents living together does not establish who lives with the student.

I would flag these zeros as potentially invalid and ask the instructor what they represent. If the definition confirms that zero is impossible, treating them as missing would be a defensible next step. Pstatus cannot tell us the correct replacement value.

---

### Parental education and occupation

Parental education and occupation are examined together to identify
common combinations and records requiring further investigation.

In [39]:
mother_education_job = pd.crosstab(
    student_data['Medu'].fillna('Missing'),
    student_data['Mjob'].fillna('Missing')
)

mother_education_job

Mjob,Missing,at_home,health,other,services,teacher
Medu,,,,,,
Higher Education,2,3,36,32,28,66
Middle School Education,3,40,5,88,43,0
Missing,1,3,0,12,8,3
None,0,3,0,3,0,0
Primary Education,0,66,2,66,6,0
Secondary Education,0,20,5,53,50,2


In [40]:
father_education_job = pd.crosstab(
    student_data['Fedu'].fillna('Missing'),
    student_data['Fjob'].fillna('Missing')
)

father_education_job

Fjob,Missing,at_home,health,other,services,teacher
Fedu,,,,,,
Higher Education,1,3,16,43,27,33
Middle School Education,2,14,2,127,58,0
Missing,1,4,2,7,4,1
None,1,1,0,5,0,0
Primary Education,7,13,0,107,42,1
Secondary Education,6,6,2,69,43,1


In [41]:
parental_status_guardian = pd.crosstab(
    student_data['Pstatus'].fillna('Missing'),
    student_data['guardian'].fillna('Missing')
)

parental_status_guardian

guardian,Missing,father,mother,other
Pstatus,,,,
A,0,6,62,9
Missing,0,10,16,2
T,8,134,372,30


In [42]:
family_size_parental_status = pd.crosstab(
    student_data['famsize'].astype('string').fillna('Missing'),
    student_data['Pstatus'].fillna('Missing')
)

family_size_parental_status

Pstatus,A,Missing,T
famsize,,,
0.0,13,3,41
1.0,15,2,31
2.0,10,1,27
3.0,9,2,37
4.0,4,0,75
5.0,3,4,81
6.0,8,2,52
7.0,5,1,61
8.0,7,5,68


In [43]:
family_size_guardian = pd.crosstab(
    student_data['famsize'].astype('string').fillna('Missing'),
    student_data['guardian'].fillna('Missing')
)

family_size_guardian

guardian,Missing,father,mother,other
famsize,,,,
0.0,2,13,40,2
1.0,1,6,36,5
2.0,0,12,26,0
3.0,0,11,33,4
4.0,1,22,55,1
5.0,1,18,62,7
6.0,2,11,46,3
7.0,0,13,48,6
8.0,1,17,57,5


### Potential inconsistencies in family-related variables

The comparisons identify several combinations requiring investigation.

- **Family size of zero — 57 records.** If family size includes the
  student, zero is invalid.
- **Mother recorded as a teacher with secondary education — 2 records.**
  The education and occupation definitions require clarification.
- **Father recorded as a teacher with secondary education — 1 record.**
  This combination also requires clarification.
- **Father recorded as a teacher with primary education — 1 record.**
  This is a particularly questionable education–occupation combination.

Other combinations depend on household arrangements.

- **Family size of one with parents living together — 31 records.**
  This may be plausible if the student lives separately from the parents.
- **Parents living together with another person as guardian — 30 records.**
  This may reflect a valid guardianship arrangement.

These records are flagged for investigation. The comparisons alone
cannot establish which values require correction or their replacements.

---

### Previous failures and absences

The distributions of previous failures and absences are inspected,
including missing entries, to investigate recorded values and the
possible meaning of blanks before deciding on a treatment.

In [44]:
student_data['failures'].value_counts(dropna=False).sort_index()

failures
1.0     69
2.0     16
3.0     13
NaN    551
Name: count, dtype: int64

In [45]:
student_data['absences'].value_counts(dropna=False).sort_index()

absences
1.0      10
2.0     108
3.0       7
4.0      88
5.0      12
6.0      46
7.0       2
8.0      41
9.0       6
10.0     21
11.0      5
12.0     12
13.0      1
14.0      8
15.0      2
16.0      9
18.0      3
21.0      2
22.0      2
24.0      1
26.0      1
30.0      1
32.0      1
NaN     260
Name: count, dtype: int64

### Treatment of missing absences and failures

A working copy of the dataset is created for preprocessing.
Missing absences and previous failures are replaced with zero,
assuming that blanks represent no absences or failures.
This interpretation remains subject to source confirmation.

In [46]:
student_data_clean = student_data.copy()

student_data_clean['absences'] = student_data_clean['absences'].fillna(0)
student_data_clean['failures'] = student_data_clean['failures'].fillna(0)

In [47]:
replacement_check = pd.DataFrame({
    'Missing before': student_data[['absences', 'failures']].isna().sum(),
    'Missing after': student_data_clean[['absences', 'failures']].isna().sum(),
    'Zeros after': student_data_clean[['absences', 'failures']].eq(0).sum()
})

replacement_check

,Missing before,Missing after,Zeros after
absences,260,0,260
failures,551,0,551


---

### Travel time, residence, and school

Travel-time distributions are compared across residence types and
schools. Zero-duration journeys are inspected to identify patterns
requiring clarification.

In [48]:
travel_summary = student_data_clean.groupby(
    ['school', 'address'],
    dropna=False
)['traveltime'].describe()

travel_summary

count       mean        std   min   25%   50%    75%    max
school address                                                             
GP     R         71.0  25.718310  30.656243   0.0   7.5  19.0  28.00  144.0
       U        332.0  13.515060  16.688810   0.0   5.0   9.0  17.00  142.0
       NaN       11.0  16.363636   9.992725   0.0  11.0  15.0  23.50   30.0
MS     R        114.0  27.315789  21.896101   0.0  16.0  25.0  33.75  135.0
       U        103.0  18.427184  13.743877   0.0   9.5  17.0  25.00   87.0
       NaN        6.0  15.666667   7.420692   7.0  10.0  15.0  20.75   26.0
NaN    U          1.0  21.000000        NaN  21.0  21.0  21.0  21.00   21.0

In [49]:
zero_travel = student_data_clean.loc[
    student_data_clean['traveltime'] == 0,
    ['school', 'address', 'traveltime']
]

zero_travel.groupby(
    ['school', 'address'],
    dropna=False
).size()

school  address
GP      R           1
        U          24
        NaN         1
MS      R           2
        U           1
dtype: int64

### Interpretation of zero travel times

There are 29 recorded zero-minute journeys, including 24 among urban
students at GP, 1 among urban students at MS, 3 among rural students,
and 1 with missing residence information.

The concentration among urban GP students is compatible with very
short journeys rounded to zero. A school-specific recording practice
is another possible explanation. The rural cases require further review.

The zeros are provisionally retained while their meaning is clarified.

---

### Study-time distribution

Weekly study time is inspected to identify zero values and unusually
high observations requiring further investigation.

In [50]:
student_data_clean['studytime'].describe(
    percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]
)

count     635.000000
mean      242.206299
std       235.969842
min         0.000000
25%        99.500000
50%       186.000000
75%       282.500000
95%       680.000000
99%      1305.920000
max      1465.000000
Name: studytime, dtype: float64

### Zero study time and academic performance

Recorded grades and previous failures are examined for students
reporting zero weekly study time to provide context for these values.

In [51]:
zero_study = student_data_clean.loc[
    student_data_clean['studytime'] == 0,
    ['studytime', 'failures', 'G1', 'G2', 'G3']
]

zero_study

,studytime,failures,G1,G2,G3
61,0.0,0.0,10,10,16
128,0.0,0.0,14,14,15
161,0.0,0.0,9,9,8


### Highest recorded study times

The ten highest weekly study times are examined alongside grades
to assess their plausibility and identify patterns for further review.

In [52]:
highest_study = student_data_clean.sort_values(
    by='studytime',
    ascending=False
)[['studytime', 'failures', 'G1', 'G2', 'G3']].head(10)

highest_study

,studytime,failures,G1,G2,G3
66,1465.0,0.0,11,12,12
503,1439.0,0.0,14,14,14
298,1368.0,0.0,10,11,12
312,1366.0,0.0,16,16,16
149,1339.0,0.0,9,11,11
343,1336.0,0.0,16,17,17
289,1310.0,0.0,14,15,17
349,1298.0,0.0,18,18,18
218,1270.0,0.0,13,13,13
106,1234.0,0.0,10,10,10
